In [50]:
import os
import shutil
import random


DATASET_PATH = "images\images_normalized\div_images"  

<>:6: SyntaxWarning: invalid escape sequence '\i'
<>:6: SyntaxWarning: invalid escape sequence '\i'
C:\Users\ag331\AppData\Local\Temp\ipykernel_23328\784376765.py:6: SyntaxWarning: invalid escape sequence '\i'
  DATASET_PATH = "images\images_normalized\div_images"


In [51]:
import pandas as pd

# تحميل البيانات
file_path = "standardized_reports_final (1).csv"  # استبدلها بمسار الملف الفعلي
df = pd.read_csv(file_path)

# حساب عدد مرات تكرار كل uid
uid_counts = df["uid"].value_counts()

# عرض التكرارات
print(uid_counts)


uid
3491    2
1       2
2       2
3       2
4       2
       ..
11      2
10      2
9       2
8       2
7       2
Name: count, Length: 3350, dtype: int64


In [52]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import os
import cv2
import numpy as np

In [53]:
# Define dataset paths
TRAIN_PATH = "images/images_normalized/div_images/train"  # Replace with your actual training folder path
VALID_PATH = "images/images_normalized/div_images/valid"  # Replace with your actual validation folder path
TEST_PATH = "images/images_normalized/div_images/test"    # Replace with your actual testing folder path


In [54]:
print(f"📂 Train images found: {len(train_files)}")
print(f"📂 Validation images found: {len(valid_files)}")
print(f"📂 Test images found: {len(test_files)}")


📂 Train images found: 4700
📂 Validation images found: 937
📂 Test images found: 651


In [55]:
# Define transformations
transform = transforms.Compose([
    transforms.ToPILImage(),  # Convert numpy image to PIL image
    transforms.Resize((224, 224)),  # Resize to match DenseNet-121 input size
    transforms.Grayscale(num_output_channels=3),  # Convert grayscale to RGB
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize to [0,1] range
])

In [56]:
# Custom dataset class
class CXRDataset(Dataset):
    def __init__(self, image_paths, transform=None): 
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):  
        return len(self.image_paths)

    def __getitem__(self, idx):  
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Read as grayscale

        if image is None:  # Handle missing or corrupt images
            print(f"❌ Warning: Image {img_path} could not be loaded!")
            return torch.zeros((3, 224, 224))  # Return a blank tensor to prevent crashes

        image = np.stack([image] * 3, axis=-1)  # Convert grayscale to 3-channel RGB
        
        if self.transform:
            image = self.transform(image)

        return image  # Return the image tensor


In [57]:

# Load image file paths
def get_image_files(directory):
    return [os.path.join(directory, file) for file in os.listdir(directory) if file.endswith((".png", ".jpg", ".jpeg"))]

train_files = get_image_files(TRAIN_PATH)
valid_files = get_image_files(VALID_PATH)
test_files = get_image_files(TEST_PATH)

In [58]:
# Create datasets
train_dataset = CXRDataset(train_files, transform=transform)
valid_dataset = CXRDataset(valid_files, transform=transform)
test_dataset = CXRDataset(test_files, transform=transform)

In [59]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [60]:
# Load pre-trained DenseNet-121 model
densenet = models.densenet121(pretrained=True)
densenet.classifier = nn.Identity()  # Remove the classification layer to get feature maps

# Set model to evaluation mode
densenet.eval()

c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [61]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
densenet.to(device)

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [62]:
def extract_features(loader, model, device):
    feature_vectors = []
    with torch.no_grad():
        for batch_idx, images in enumerate(loader):
            print(f"🔄 Processing batch {batch_idx + 1}, Batch size: {images.shape[0]}")

            images = images.to(device)
            features = model(images)  # Extract features from DenseNet-121
            feature_vectors.append(features.cpu().numpy())  # Move to CPU for further processing

    return np.vstack(feature_vectors)  # Convert list to NumPy array


In [63]:

# Extract features for train, validation, and test sets
train_features = extract_features(train_loader, densenet, device)
valid_features = extract_features(valid_loader, densenet, device)
test_features = extract_features(test_loader, densenet, device)

🔄 Processing batch 1, Batch size: 32
🔄 Processing batch 2, Batch size: 32
🔄 Processing batch 3, Batch size: 32
🔄 Processing batch 4, Batch size: 32
🔄 Processing batch 5, Batch size: 32
🔄 Processing batch 6, Batch size: 32
🔄 Processing batch 7, Batch size: 32
🔄 Processing batch 8, Batch size: 32
🔄 Processing batch 9, Batch size: 32
🔄 Processing batch 10, Batch size: 32
🔄 Processing batch 11, Batch size: 32
🔄 Processing batch 12, Batch size: 32
🔄 Processing batch 13, Batch size: 32
🔄 Processing batch 14, Batch size: 32
🔄 Processing batch 15, Batch size: 32
🔄 Processing batch 16, Batch size: 32
🔄 Processing batch 17, Batch size: 32
🔄 Processing batch 18, Batch size: 32
🔄 Processing batch 19, Batch size: 32
🔄 Processing batch 20, Batch size: 32
🔄 Processing batch 21, Batch size: 32
🔄 Processing batch 22, Batch size: 32
🔄 Processing batch 23, Batch size: 32
🔄 Processing batch 24, Batch size: 32
🔄 Processing batch 25, Batch size: 32
🔄 Processing batch 26, Batch size: 32
🔄 Processing batch 27

In [67]:
# Save extracted features
np.save("train_features.npy", train_features)
np.save("valid_features.npy", valid_features)
np.save("test_features.npy", test_features)

print("Feature extraction complete!")
print(f"Train features shape: {train_features.shape}")  # Expected: (2500, 1024)
print(f"Validation features shape: {valid_features.shape}")  # Expected: (500, 1024)
print(f"Test features shape: {test_features.shape}")  # Expected: (350, 1024)

Feature extraction complete!
Train features shape: (4700, 1024)
Validation features shape: (937, 1024)
Test features shape: (651, 1024)


In [ ]:
import pandas as pd

# Load the dataset
file_path = "indiana_projections.csv"  # Replace with your actual file path
df = pd.read_csv(file_path)

# Group by `uid` to get all images associated with each report
grouped = df.groupby("uid")

# List to store the final standardized dataset
standardized_data = []

# Function to process each report
def process_report(uid, images):
    frontal_images = [img for img, proj in images if proj == "Frontal"]
    lateral_images = [img for img, proj in images if proj == "Lateral"]

    selected_images = []

    if len(frontal_images) == 1 and len(lateral_images) == 1:
        # ✅ Keep reports with exactly two images (1 Frontal + 1 Lateral)
        selected_images = [frontal_images[0], lateral_images[0]]

    elif len(frontal_images) == 0 or len(lateral_images) == 0:
        # ❌ Duplicate the single image if only one exists
        single_image = frontal_images[0] if frontal_images else lateral_images[0]
        selected_images = [single_image, single_image]

    elif len(frontal_images) + len(lateral_images) == 3:
        # ✂ Pick one Frontal and one Lateral, discard the extra
        selected_images = [frontal_images[0], lateral_images[0]]

    elif len(frontal_images) >= 2 and len(lateral_images) >= 2:
        # ✂ Pick one Frontal and one Lateral from four-image reports
        selected_images = [frontal_images[0], lateral_images[0]]

    # Store selected images in the standardized dataset
    for img in selected_images:
        standardized_data.append({"uid": uid, "filename": img})

# Apply the function to all reports
for uid, group in grouped:
    images = list(zip(group["filename"], group["projection"]))  # Get image names and types
    process_report(uid, images)

# Convert list to DataFrame
standardized_df = pd.DataFrame(standardized_data)

# Save to new CSV file
output_csv = "standardized_reports.csv"
standardized_df.to_csv(output_csv, index=False)

print(f"✅ Standardization complete: All reports now have exactly 2 images each. Saved to {output_csv}")


✅ Standardization complete: All reports now have exactly 2 images each. Saved to standardized_reports.csv
